In [ ]:
%%capture
!pip install pycaret[full]

In [ ]:
import pandas as pd
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 

import gc

from pycaret.regression import *

import cudf

%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = cudf.read_csv('../input/tabular-playground-series-jan-2022/train.csv', index_col = 'row_id').to_pandas()
test = cudf.read_csv('../input/tabular-playground-series-jan-2022/test.csv', index_col = 'row_id').to_pandas()

In [ ]:
# Credit to https://www.kaggle.com/ranjeetshrivastav/tps-jan-21-base-xgb

train['date'] = pd.to_datetime(train['date'])
train['year'] = train['date'].dt.year
train['month'] = train['date'].dt.month
train['day'] = train['date'].dt.day
train['dayofweek'] = train['date'].dt.dayofweek
train['dayofmonth'] = train['date'].dt.days_in_month
train['dayofyear'] = train['date'].dt.dayofyear
train['weekday'] = train['date'].dt.weekday

test['date'] = pd.to_datetime(test['date'])
test['year'] = test['date'].dt.year
test['month'] = test['date'].dt.month
test['day'] = test['date'].dt.day
test['dayofweek'] = test['date'].dt.dayofweek
test['dayofmonth'] = test['date'].dt.days_in_month
test['dayofyear'] = test['date'].dt.dayofyear
test['weekday'] = test['date'].dt.weekday

train.drop('date', axis = 1, inplace = True)
test.drop('date', axis = 1, inplace = True)

In [ ]:
reg = setup(data = train,
            target = 'num_sold',
            data_split_shuffle = False,
            #create_clusters = True,
            use_gpu = True,
            silent = True,
            n_jobs = -1)

In [ ]:
models()

In [ ]:
compare_models(sort = 'MAPE')

In [ ]:
N = 7
top = compare_models(sort='MAPE', n_select=N)

In [ ]:
blend = blend_models(top)
predict_model(blend)

In [ ]:
final_blend = finalize_model(blend)
predict_model(final_blend)

In [ ]:
#tuned_top = [tune_model(i, optimize = 'MAPE', choose_better = True) for i in top]

In [ ]:
#tuned_blend = blend_models(tuned_top)
#predict_model(tuned_blend);

In [ ]:
#final_tuned_blend = finalize_model(tuned_blend)
#predict_model(final_tuned_blend);

In [ ]:
gc.collect()
unseen_predictions_blend = predict_model(final_blend, data=test)
unseen_predictions_blend.head()

In [ ]:
gc.collect()

assert(len(test.index)==len(unseen_predictions_blend))

sub = pd.DataFrame(list(zip(test.index, unseen_predictions_blend.Label)),columns = ['row_id', 'num_sold'])

sub.to_csv('submission.csv', index = False)

print(sub)